<a href="https://colab.research.google.com/github/calmrocks/ai-engineer-notebooks/blob/main/00-setup/00-environment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Environment & Cost Hygiene

**Goal:** Set up API keys via Colab secrets, add spend guards, and make one successful model call.

Part of [ai-engineer-notebooks](https://github.com/calmrocks/ai-engineer-notebooks) — the hands-on companion to the [FDE / AI Engineer transition plan](https://www.calm.rocks/resources/career-development/transition-fde-ai-engineer/).


In [ ]:
%pip install -q groq

## API keys via Colab Secrets

Never paste an API key into a notebook cell. Notebooks get shared, committed, and screenshotted — a key in a cell is a key you'll be rotating next week.

Get your free Groq API key at **console.groq.com** — no credit card required.

Then add it to Colab:

1. Click the **key icon** in the left sidebar ("Secrets").
2. Click **Add new secret**. Name it exactly `GROQ_API_KEY`, paste your key as the value.
3. Flip the **Notebook access** toggle on for this notebook.

The cell below reads that secret in Colab, or falls back to the `GROQ_API_KEY` environment variable if you're running locally.


In [ ]:
import os

# In Colab, read the key from Secrets (key icon in the left sidebar).
# Locally, set the GROQ_API_KEY env var instead.
try:
    from google.colab import userdata
    os.environ['GROQ_API_KEY'] = userdata.get('GROQ_API_KEY')
except ImportError:
    assert os.environ.get('GROQ_API_KEY'), 'Set GROQ_API_KEY'

from groq import Groq
client = Groq()
MODEL = 'llama-3.3-70b-versatile'  # good default: capable and free on Groq

## First call

One request, one response. Groq follows the OpenAI-compatible format: `response.choices[0].message.content` is the text, and `response.usage` is the billing record for this call.


In [ ]:
response = client.chat.completions.create(
    model=MODEL,
    max_tokens=200,
    messages=[{'role': 'user', 'content': 'In two sentences: what does a forward-deployed engineer do?'}],
)

print(response.choices[0].message.content)
print()
print('finish_reason:', response.choices[0].finish_reason)
print('usage:', response.usage)


## Reading `response.usage`

Every response carries the exact token accounting:

- `prompt_tokens` — input tokens (your messages)
- `completion_tokens` — output tokens (what the model generated)
- `total_tokens` — sum of both

Groq does not have prompt caching, so there are no cache fields. For cost, multiply each bucket by its rate. That's all a spend guard needs.


## A `spend_guard` for the session

During development you'll re-run cells, loop over datasets, and occasionally fat-finger a loop that makes 500 calls instead of 5. A small wrapper that accumulates estimated cost and refuses to go past a session budget turns that mistake from a bill into an exception.

Prices below are the public per-million-token rates as of mid-2026 — they drift, so treat the table as something you update, not a constant.


In [ ]:
PRICES = {
    # USD per million tokens: (input, output). Groq free tier has no cost — use these for
    # comparison exercises or if you move to a paid tier. Check groq.com/pricing.
    'llama-3.3-70b-versatile': (0.59, 0.79),
    'llama-3.1-8b-instant':    (0.05, 0.08),
    'llama3-70b-8192':         (0.59, 0.79),
}


class BudgetExceeded(Exception):
    pass


class SpendGuard:
    """Wraps client.chat.completions.create, accumulates estimated cost, raises past a budget."""

    def __init__(self, budget_usd=0.50):
        self.budget = budget_usd
        self.spent = 0.0
        self.calls = 0

    def _cost(self, model, usage):
        inp_rate, out_rate = PRICES.get(model, (0.59, 0.79))
        return (usage.prompt_tokens * inp_rate + usage.completion_tokens * out_rate) / 1_000_000

    def create(self, **kwargs):
        if self.spent >= self.budget:
            raise BudgetExceeded(
                f'Session spend ${self.spent:.4f} >= budget ${self.budget:.2f} — refusing to call.'
            )
        response = client.chat.completions.create(**kwargs)
        cost = self._cost(kwargs['model'], response.usage)
        self.spent += cost
        self.calls += 1
        print(
            f'[spend_guard] call {self.calls}: '
            f'{response.usage.prompt_tokens} in / {response.usage.completion_tokens} out '
            f'~= ${cost:.5f}  (session: ${self.spent:.5f} / ${self.budget:.2f})'
        )
        return response


guard = SpendGuard(budget_usd=0.25)


In [ ]:
# Normal use: same signature as client.chat.completions.create, plus a running tally.
resp = guard.create(
    model=MODEL,
    max_tokens=100,
    messages=[{'role': 'user', 'content': 'One sentence: why do teams add spend guards around LLM calls?'}],
)
print(resp.choices[0].message.content)


In [ ]:
# And the failure path: a deliberately tiny budget trips after the first call.
tiny = SpendGuard(budget_usd=0.0001)
try:
    tiny.create(model=MODEL, max_tokens=50,
                messages=[{'role': 'user', 'content': 'Say hi.'}])  # this one succeeds...
    tiny.create(model=MODEL, max_tokens=50,
                messages=[{'role': 'user', 'content': 'Say hi again.'}])  # ...this one raises
except BudgetExceeded as e:
    print('Guard tripped as expected:', e)


Run the two cells above and note that the guard checks *before* calling — the second call never leaves your machine. In a real service you'd hang this off a per-tenant or per-day counter in Redis rather than an in-memory object, but the shape is identical: meter from `response.usage`, gate before the request.


## Picking a model (on Groq)

The names will change; the principle won't. Two main tiers available free on Groq:

- **70B-class** (`llama-3.3-70b-versatile`) — the iterate-on default. Strong enough for almost everything, and fast on Groq's hardware.
- **8B-class** (`llama-3.1-8b-instant`) — bulk and cheap operations: classification, extraction over thousands of rows, judge calls in eval loops, anything where you multiply by N.

The working rule: **develop on cheap, eval on target.** Build the pipeline against the 8B model, then run your evaluation set against the 70B model you'll actually ship before you commit. Prompt behavior shifts between tiers — a prompt tuned on 8B may behave differently on 70B — so the eval-on-target step is not optional.


In [ ]:
# Same prompt on two tiers — compare cost and feel. (2 API calls.)
prompt = 'Classify the sentiment of this review as positive, negative, or mixed: '\
         '"The battery life is great but the screen scratched in a week." Reply with one word.'

for model in ['llama-3.1-8b-instant', 'llama-3.3-70b-versatile']:
    r = guard.create(model=model, max_tokens=10,
                     messages=[{'role': 'user', 'content': prompt}])
    print(f'{model}: {r.choices[0].message.content.strip()}')


Run it and note that both tiers nail a task this simple — which is exactly the point. If the 8B model passes your eval for a task, routing it to the 70B is just burning margin.


## Exercises

1. Extend `SpendGuard.create` to record per-call wall-clock latency (`time.monotonic()` around the request) and add a `.report()` method that prints total cost, total calls, and p50/max latency.
2. Take a prompt you care about and phrase it two ways — terse bullet-point style vs. full prose. Compare `prompt_tokens` for each (send both with `max_tokens=1` to keep it cheap) and compute the cost difference at 100k calls/month.
3. Add a `hard_max_tokens` option to `SpendGuard` that rejects any call requesting more than N output tokens, so a typo like `max_tokens=100000` can't slip through.
4. The free Groq tier has rate limits (requests/minute and tokens/minute). Add a retry loop to `SpendGuard.create` that catches `groq.RateLimitError` and backs off with `time.sleep`.
